In [2]:
!pip install requests folium pandas geopandas -q 

In [32]:
#@title Importar GeoPandas
import geopandas as gpd
import pandas as pd
import requests
import numpy as np
import requests
import io
import folium
from folium.plugins import HeatMap


##  Análise de ilícitos ambientais - Ibama
![](https://wikilai.fiquemsabendo.com.br/images/thumb/a/a8/Logo_principal_-_fiquem_sabendo.png/320px-Logo_principal_-_fiquem_sabendo.png)

# Importação dos Dados Ibama e TSE

In [5]:
#@title GeoJson dos Embargos do Ibama maxfeatures = 100000
url_embargos_ibama ='https://siscom.ibama.gov.br/geoserver/publica/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=publica:vw_brasil_adm_embargo_a&maxFeatures=100000&outputFormat=application%2Fjson'


In [6]:
#@title Ler GeoJson como GeoDataFrame
# Ler GeoDataFrame
# Adicionar um cabeçalho User-Agent para evitar o erro 403 Forbidden
# A URL do IBAMA pode bloquear requisições sem um User-Agent ou com User-Agents genéricos.

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

# Fazer a requisição HTTP usando requests
response = requests.get(url_embargos_ibama, headers=headers)
response.raise_for_status() # Lança uma exceção para erros HTTP (4xx ou 5xx)

# Usar io.BytesIO para tratar o conteúdo da resposta como um arquivo para o geopandas
gdf = gpd.read_file(io.BytesIO(response.content))

# Converter colunas para minúscula
gdf.columns = gdf.columns.str.lower()

In [23]:
gdf.columns

Index(['id', 'nom_pessoa', 'cpf_cnpj_infrator', 'seq_tad', 'numero_tad',
       'data_tad', 'num_latitude_tad', 'num_longitude_tad', 'processo_tad',
       'sig_uf', 'nom_municipio', 'num_auto_infracao', 'ser_auto_infracao',
       'cod_municipio', 'cod_uf', 'status_tad', 'qtd_area_desmatada',
       'data_cadastro_tad', 'des_infracao', 'serie_tad',
       'sit_embarga_poligono', 'legislacao', 'artigo_legislacao', 'artigo',
       'imagem_validacao', 'respeita_embargo', 'data_geom', 'orgao',
       'geometry', 'ano_tad'],
      dtype='str')

In [28]:
gdf['cpf_cnpj_infrator'].value_counts()

cpf_cnpj_infrator
054.408.922-73        191
009.150.172-53         25
04.680.054/0001-04     23
340.307.861-20         21
635.748.964-68         20
                     ... 
987.963.057-20          1
056.574.768-16          1
076.812.593-68          1
709.412.117-20          1
140.688.184-87          1
Name: count, Length: 66335, dtype: int64

In [22]:

DadosAno = gdf[gdf['ano_tad'].between(2018, 2024)]


In [29]:
DadosAno['cpf_cnpj_infrator'].value_counts()

cpf_cnpj_infrator
021.554.008-53        18
04.680.054/0001-04    18
010.720.611-04        14
001.294.103-49        14
18.336.502/0001-53    13
                      ..
188.975.179-00         1
776.498.189-34         1
038.457.182-42         1
557.418.783-34         1
465.562.067-68         1
Name: count, Length: 18407, dtype: int64

In [38]:
import requests, zipfile, io

# Exemplo: candidatos de 2022
url = "https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2022.zip"

# Baixar e abrir o zip
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Ver arquivos disponíveis
print(z.namelist())

# Ler um CSV específico
df = pd.read_csv(z.open("consulta_cand_2022_BRASIL.csv"), sep=";", encoding="latin1")



['leiame.pdf', 'consulta_cand_2022_CE.csv', 'consulta_cand_2022_AL.csv', 'consulta_cand_2022_AC.csv', 'consulta_cand_2022_BA.csv', 'consulta_cand_2022_AM.csv', 'consulta_cand_2022_AP.csv', 'consulta_cand_2022_MA.csv', 'consulta_cand_2022_MT.csv', 'consulta_cand_2022_RO.csv', 'consulta_cand_2022_GO.csv', 'consulta_cand_2022_MS.csv', 'consulta_cand_2022_ES.csv', 'consulta_cand_2022_MG.csv', 'consulta_cand_2022_PB.csv', 'consulta_cand_2022_DF.csv', 'consulta_cand_2022_PA.csv', 'consulta_cand_2022_PR.csv', 'consulta_cand_2022_PE.csv', 'consulta_cand_2022_PI.csv', 'consulta_cand_2022_RJ.csv', 'consulta_cand_2022_RN.csv', 'consulta_cand_2022_SC.csv', 'consulta_cand_2022_RS.csv', 'consulta_cand_2022_RR.csv', 'consulta_cand_2022_SP.csv', 'consulta_cand_2022_TO.csv', 'consulta_cand_2022_BR.csv', 'consulta_cand_2022_SE.csv', 'consulta_cand_2022_BRASIL.csv']


In [35]:
df.columns

Index(['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'CD_TIPO_ELEICAO',
       'NM_TIPO_ELEICAO', 'NR_TURNO', 'CD_ELEICAO', 'DS_ELEICAO', 'DT_ELEICAO',
       'TP_ABRANGENCIA', 'SG_UF', 'SG_UE', 'NM_UE', 'CD_CARGO', 'DS_CARGO',
       'SQ_CANDIDATO', 'NR_CANDIDATO', 'NM_CANDIDATO', 'NM_URNA_CANDIDATO',
       'NM_SOCIAL_CANDIDATO', 'NR_CPF_CANDIDATO', 'DS_EMAIL',
       'CD_SITUACAO_CANDIDATURA', 'DS_SITUACAO_CANDIDATURA', 'TP_AGREMIACAO',
       'NR_PARTIDO', 'SG_PARTIDO', 'NM_PARTIDO', 'NR_FEDERACAO',
       'NM_FEDERACAO', 'SG_FEDERACAO', 'DS_COMPOSICAO_FEDERACAO',
       'SQ_COLIGACAO', 'NM_COLIGACAO', 'DS_COMPOSICAO_COLIGACAO',
       'SG_UF_NASCIMENTO', 'DT_NASCIMENTO', 'NR_TITULO_ELEITORAL_CANDIDATO',
       'CD_GENERO', 'DS_GENERO', 'CD_GRAU_INSTRUCAO', 'DS_GRAU_INSTRUCAO',
       'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_COR_RACA', 'DS_COR_RACA',
       'CD_OCUPACAO', 'DS_OCUPACAO', 'CD_SIT_TOT_TURNO', 'DS_SIT_TOT_TURNO'],
      dtype='str')

In [39]:
df.shape

(29314, 50)